# 💻 Architecture & Code : Intégration du LLM dans SalesTeam AI

Ce notebook documente **l'intégration complète de l'IA Générative (LLM)** réalisée dans le projet. Il détaille les fichiers modifiés, les nouvelles fonctions créées, et la logique globale de l'architecture pour générer des explications fiables, rapides et sans hallucination.

## 1. La Couche Données : Le Contexte Profond

**📁 Fichier :** `src/services/deep_context.py` *(Nouveau fichier)*

Pour éviter que l'IA n'invente des chiffres (hallucinations), nous avons créé un service dédié à la récolte de **preuves vérifiables**.

* **`retrieve_deep_context(client_id, code_article, response_cache)`** : C'est la fonction maîtresse. Elle assemble un dictionnaire de données contenant :
  1. L'historique réel (les 6 dernières commandes exactes extraites de `main_table.csv`).
  2. Les *features* dynamiques (comme la récence exacte calculée à la date de visite).
  3. La décomposition du score ML.
* **Le mécanisme de Feature dynamique** : Dans `_get_feature_values()`, on récupère d'abord les valeurs calculées en temps réel dans l'objet `ProductSuggestion` (qui prend en compte la date choisie par l'utilisateur). Si elles sont absentes, on utilise la base statique `training_set.csv` en fallback.

## 2. La Couche IA : Prompt Engineering & Fallback

**📁 Fichier :** `src/services/explanation.py`

C'est ici que l'on communique avec le LLM (Llama 3.3 via HuggingFace).

* **`explain_suggestion_detailed(context)`** : Construit le grand prompt (texte) envoyé à l'IA. Elle impose 3 règles strictes au modèle : une structure en 4 parties (`Pourquoi ce produit ?`, etc.), l'interdiction de citer des chiffres absents du contexte, et l'interdiction d'utiliser du jargon technique.
* **`_call_huggingface_api()` et `_quota_exhausted`** : Gère l'appel réseau. J'y ai ajouté un **Circuit Breaker**. Si l'API répond `HTTP 402` (Quota épuisé), le booléen `_quota_exhausted` passe à `True` et bloque immédiatement tous les futurs appels HTTP pour éviter de spammer le serveur et planter l'application.
* **`_rule_based_detailed_explanation()`** : Le fallback. Si le LLM échoue (quota, réseau), cette fonction prend le même dictionnaire de contexte et génère les 4 paragraphes en utilisant du code Python pur. Elle garantit que le système ne tombe jamais en panne.

## 3. La Couche Métier : Le Mécanisme de Cache et de Préservation

**📁 Fichier :** `src/services/recommendation.py`

L'appel à un LLM coûte cher en temps et en quota.

* **`_last_response_cache`** : Un dictionnaire en mémoire qui sauvegarde le dernier résultat généré par `/api/recommend` pour chaque client. Ainsi, quand on demande une explication détaillée, on ne refait pas tourner XGBoost.
* **`get_detailed_explanation()`** : La fonction appelée par le endpoint API. Elle lit le cache, extrait l'article cliqué, invoque `retrieve_deep_context()`, puis appelle le générateur d'explication.
* **L'astuce `_skip_llm=True`** : Quand on calcule les recommandations de base (l'affichage des 12 cartes), on appelle `recommend()` avec `_skip_llm=True`. Cela force l'utilisation du fallback (rapide et gratuit) pour les 12 petites descriptions, préservant ainsi la totalité du quota HuggingFace pour l'unique moment où le client clique sur une carte pour voir les détails.

## 4. La Couche API : L'Exposition au Frontend

**📁 Fichier :** `src/api/schemas.py`

Nous avons enrichi les structures de données Pydantic :
* Ajout des champs dynamiques (`recency_days`, `trend`, `avg_delay_days`, `frequency`) dans **`ProductSuggestion`** pour qu'ils soient sauvés dans le cache en tenant compte de la vraie date de visite sélectionnée dans l'interface.
* Création de **`DetailedExplanationRequest`** et **`DetailedExplanationResponse`** pour normaliser le flux de l'explication.

**📁 Fichier :** `src/api/routes/recommend.py`

* Ajout de la route **`POST /api/explain-detailed`**. Contrairement à `/api/recommend` (qui traite tout un client), cette route est ciblée sur un unique couple `(client_id, code_article)`.

## 5. La Couche Interface : L'Expérience Utilisateur

**📁 Fichier :** `frontend/src/App.jsx` & `frontend/src/index.css`

Il ne servait à rien d'avoir une explication générée si on ne pouvait pas la lire.

* **La Modale (Pop-up)** : Modifiée en CSS pour être plus large (`750px`), redimensionnable dynamiquement par la souris (`resize: both;`), et intégrant une barre de défilement interne (`overflow-y: auto;`) pour que le texte respire.
* **Appel à la demande** : Mise en place d'un `useEffect` dans React. La route `/explain-detailed` n'est appelée **qu'au moment exact** où l'utilisateur clique sur la carte.
* **Le Parseur de Markdown (`DetailedSections`)** : Création d'un composant React qui lit la réponse Markdown (les `## Pourquoi...`), découpe le texte, et l'affiche sous forme de 4 blocs séparés avec des bordures et des couleurs différentes pour une lisibilité immédiate.

## ⚡ Résumé de la cinématique (Le Flux de Données)

1. L'utilisateur lance une recherche -> `POST /api/recommend` calcule le ML et renvoie 12 cartes **instantanément** sans utiliser l'API HuggingFace (`_skip_llm=True`). Le résultat est stocké dans la RAM du serveur (`_last_response_cache`).
2. L'utilisateur clique sur la carte N°1 -> La modale s'ouvre, affiche le *spinner* "Analyse en cours..." et le frontend lance `POST /api/explain-detailed`.
3. Le backend lit le cache, envoie le `client_id` et le `code_article` à `retrieve_deep_context()` qui extrait l'historique.
4. `explain_suggestion_detailed()` envoie cet historique au LLM.
5. Le LLM génère l'explication, le backend la renvoie au frontend.
6. Le composant `DetailedSections` découpe le texte en 4 blocs colorés et les affiche dans la grande popup.